In [ ]:
# import torch and other necessary modules from torch
import torch
import torch.nn as nn

...
# import torchvision and other necessary modules from torchvision
import torchvision
from torchvision import transforms
from torchvision import datasets
...
torch.manual_seed(42)

# recommended preprocessing steps: resize to square -> convert to tensor -> normalize the image
# if you are resizing, 100 is a good choice otherwise GradeScope will time out
# you could use Compose (https://pytorch.org/vision/stable/generated/torchvision.transforms.Compose.html)
# from transforms module to handle preprocessing more conveniently

transform = transforms.Compose([transforms.Resize((100,100)),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=0,std=1)])


In [ ]:
# thanks to torchvision, this is a convenient way to read images from folders
# directly without writing datasets class yourself (you should know what datasets
# class is as mentioned in the documentation)

import zipfile

#with zipfile.ZipFile("./petimages-1.zip", "r") as z:
#    z.extractall("petimages")

dataset = datasets.ImageFolder('./petimages', transform=transform)

In [ ]:
from torch.utils.data import random_split, DataLoader

# now we need to split the data into training set and evaluation set
# use 20% of the dataset as test
train_set, test_set = torch.utils.data.random_split(dataset,
                                                    lengths=[0.8, 0.2])

In [ ]:
# prepare dataloader for training set and evaluation set
trainloader = torch.utils.data.DataLoader(dataset=train_set, batch_size=10, shuffle=True)
testloader = torch.utils.data.DataLoader(dataset=test_set, batch_size=10, shuffle=True)

# model hyperparameter
learning_rate = 0.1
batch_size = 10
epoch_size = 10

In [ ]:
# model design goes here
class CNN(nn.Module):

  # there is no "correct" CNN model architecture for this lab, you can start with
  # a naive model as follows:
  # convolution -> relu -> pool -> convolution -> relu -> pool -> convolution ->
  # relu -> pool -> linear -> relu -> linear -> relu -> linear
  # you can try increasing number of convolution layers or try totally different model design
  # convolution: nn.Conv2d (https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
  # pool: nn.MaxPool2d (https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)
  # linear: nn.Linear(https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)

  def __init__(self):
    super(CNN,self).__init__()

    self.layer1 = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=2)
    self.maxLayer = nn.MaxPool2d(kernel_size=2)
    self.layer3 = nn.Conv2d(in_channels=8, out_channels=4, kernel_size=3)
    self.layer4 = nn.Conv2d(in_channels=4, out_channels=2, kernel_size=3)
    self.relu = nn.ReLU()

    self.linear1 = nn.Linear(in_features=200, out_features = 200)
    self.linear2 = nn.Linear(in_features = 200, out_features=50)
    self.linear3 = nn.Linear(in_features=50, out_features = 2)

  ...
  def forward(self, x):
    test = self.layer1(x)
    test = self.relu(test)
    test = self.maxLayer(test)
    test = self.layer3(test)
    test = self.relu(test)
    test = self.maxLayer(test)
    test = self.layer4(test)

    test = self.relu(test)
    test = self.maxLayer(test)

    test = test.flatten(start_dim=1)
    test = self.linear1(test)

    test = self.relu(test)
    test = self.linear2(test)
    test = self.relu(test)

    return self.linear3(test)


images, labels = next(iter(trainloader))

cnn = CNN()
x = cnn.forward(images)
print(x)

tensor([[-0.0364, -0.1168],
        [-0.0371, -0.1160],
        [-0.0383, -0.1152],
        [-0.0358, -0.1158],
        [-0.0373, -0.1173],
        [-0.0369, -0.1149],
        [-0.0363, -0.1161],
        [-0.0366, -0.1161],
        [-0.0367, -0.1154],
        [-0.0372, -0.1173]], grad_fn=<AddmmBackward0>)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu' # whether your device has GPU
cnn = CNN().to(device) # move the model to GPU

In [ ]:
# search in official website for CrossEntropyLoss
criterion = nn.CrossEntropyLoss()

# try Adam optimizer (https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) with learning
# rate 0.0001, feel free to use other optimizer

optimizer = torch.optim.Adam(params = cnn.parameters(), lr = 0.0001)

In [ ]:
# start model training
cnn.train() # turn on train mode, this is a good practice to do

for epoch in range(epoch_size): # begin with trying 10 epochs
  loss = 0.0 # you can print out average loss per batch every certain batches

  for i, (inputs, labels) in enumerate(trainloader, 0):
    # move tensors to your current device (cpu or gpu)

    inputs = inputs.to(device)
    labels = labels.to(device)

    # zero the parameter gradients using zero_grad()
    optimizer.zero_grad()

    # forward -> compute loss -> backward propogation -> optimize (see tutorial mentioned in main documentation)
    # print some statistics
    pred = cnn.forward(inputs)
    batch_loss = criterion(pred, labels)
    batch_loss.backward()
    optimizer.step()

    loss += batch_loss.item() # add loss for current batch
    if i % 100 == 0: # print out average loss every 100 batches
      print(f'[{epoch + 1}, {i + 1:5d}] loss: {loss / 100:.3f}')
      loss = 0.0

print('Finished Training')


[1,     1] loss: 0.008
[1,    41] loss: 0.283
[1,    81] loss: 0.473
[1,   121] loss: 0.537
[1,   161] loss: 0.548
[1,   201] loss: 0.551
[1,   241] loss: 0.553
[1,   281] loss: 0.554
[2,     1] loss: 0.000
[2,    41] loss: 0.000
[2,    81] loss: 0.000
[2,   121] loss: 0.001
[2,   161] loss: 0.001
[2,   201] loss: 0.001
[2,   241] loss: 0.001
[2,   281] loss: 0.001
[3,     1] loss: 0.000
[3,    41] loss: 0.000
[3,    81] loss: 0.000
[3,   121] loss: 0.000
[3,   161] loss: 0.000
[3,   201] loss: 0.000
[3,   241] loss: 0.000
[3,   281] loss: 0.000
[4,     1] loss: 0.000
[4,    41] loss: 0.000
[4,    81] loss: 0.000
[4,   121] loss: 0.000
[4,   161] loss: 0.000
[4,   201] loss: 0.000
[4,   241] loss: 0.000
[4,   281] loss: 0.000
[5,     1] loss: 0.000
[5,    41] loss: 0.000
[5,    81] loss: 0.000
[5,   121] loss: 0.000
[5,   161] loss: 0.000
[5,   201] loss: 0.000
[5,   241] loss: 0.000
[5,   281] loss: 0.000
[6,     1] loss: 0.000
[6,    41] loss: 0.000
[6,    81] loss: 0.000
[6,   121] 

In [ ]:
# evaluation on evaluation set
ground_truth = []
prediction = []

import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score

cnn.eval() # turn on evaluation model, also a good practice to do

with torch.no_grad(): # since we're not training, we don't need to calculate the
  #gradients for our outputs, so turn on no_grad mode
  for (inputs, labels) in testloader:

    inputs = inputs.to(device)
    ground_truth += list(labels)# convert labels to list and append to ground_truth
    # calculate outputs by running inputs through the network

    outputs = cnn(inputs)
    # the class with the highest logit is what we choose as prediction
    _, predicted = torch.max(outputs, dim=1)

    prediction += list(predicted) # convert predicted to list and append to prediction
    # GradeScope is chekcing for these three variables, you can use sklearn to
    # calculate the scores
accuracy = accuracy_score(ground_truth, prediction)
recall = recall_score(ground_truth, prediction, average='macro')
precision = precision_score(ground_truth, prediction, average='macro')

print('accuracy: ', accuracy)
print('recall: ', recall)
print('precision: ', precision)


accuracy:  1.0
recall:  1.0
precision:  1.0
